In [ ]:
import os
import zipfile
import io
import netrc
import requests
from tqdm import tqdm

# Change PRODUCT_NAME to the SAFE file you want to download.
PRODUCT_NAME = "S2A_MSIL1C_20250102T183751_N0511_R027_T11SLT_20250102T202910.SAFE"

# Save to the folder data/.
output_dir = os.path.join("data", PRODUCT_NAME)
os.makedirs("data", exist_ok=True)

# Check if the .SAFE directory already exists and is non-empty
if os.path.exists(output_dir) and os.listdir(output_dir):
    print(f"Skipping download: '{PRODUCT_NAME}' is already downloaded in data/.")
else:
    # 1. Fetch Product ID
    search_url = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{PRODUCT_NAME}'"
    response = requests.get(search_url).json()
    if not response.get("value"):
        raise ValueError(f"Product '{PRODUCT_NAME}' was not found.")
    product_id = response["value"][0]["Id"]
    download_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

    # 2. Authenticate
    login, _, password = netrc.netrc().authenticators(
        "identity.dataspace.copernicus.eu"
    )
    token = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            "username": login,
            "password": password,
            "grant_type": "password",
        },
    ).json()["access_token"]

    # 3. Stream and extract directly into the SAFE folder
    with requests.get(
        download_url, headers={"Authorization": f"Bearer {token}"}, stream=True
    ) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        chunk_size = 8192
        with (
            io.BytesIO() as buffer,
            tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="Downloading",
            ) as bar,
        ):
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.write(chunk)
                    bar.update(len(chunk))
            buffer.seek(0)
            with zipfile.ZipFile(buffer) as zf:
                # Extracts into data/ folder, using the root path inside the zip file
                zf.extractall("data")

    print(f"Saved to {output_dir}")